# Cautious Chainsaw: Llama experiments

1. Select a Colab GPU runtime (T4 defaults to FP16; L4/A100 default to BF16).
2. Edit Config and add Secrets as described below.
3. Run all cells. Completed CLI outputs are reused with `--resume`.

In Colab Secrets, add `GITHUB_TOKEN` (fine-grained, this repository only, **Contents: read**) and optionally `HF_TOKEN` (**read**); grant notebook access to each. A valid source folder on Drive is reused without GitHub credentials. Choose a new `RUN_NAME` for a different model or experiment configuration.


## 1. Config

In [ ]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/cautious-chainsaw")
REPO = DRIVE / "repo"
REPO_URL = "https://github.com/Mayonnaise-Slap/cautious-chainsaw.git"
REF = "main"
RUN_NAME = "llama32-1b-default"
MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"
BACKEND = "auto"  # auto, bf16, or fp16

## 2. Setup Drive, Hugging Face cache, and Secrets

In [ ]:
import os

from google.colab import drive, userdata

drive.mount("/content/drive")
CACHE = DRIVE / "hf-cache"
REPO.parent.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
os.environ.update(
    {
        "HF_HOME": str(CACHE),
        "HF_TOKEN_PATH": "/content/hf-token",
        "HF_HUB_CACHE": str(CACHE / "hub"),
        "HF_DATASETS_CACHE": str(CACHE / "datasets"),
        "TRANSFORMERS_CACHE": str(CACHE / "hub"),
        "UV_PROJECT_ENVIRONMENT": "/content/chainsaw-venv",
        "TOKENIZERS_PARALLELISM": "false",
    }
)


def optional_secret(name):
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return None


hf_token = optional_secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
else:
    os.environ.pop("HF_TOKEN", None)
    print("HF_TOKEN unavailable; public Hugging Face assets remain usable.")
del hf_token


def check_shell(label):
    status = get_ipython().user_ns.pop("_exit_code")
    if status:
        raise RuntimeError(f"{label} failed with exit status {status}.")

## 3. Check GPU and choose precision

In [ ]:
import torch

!nvidia-smi --query-gpu=name,memory.total --format=csv
check_shell("nvidia-smi")
assert torch.cuda.is_available(), "Select a Colab GPU runtime."
native_bf16 = torch.cuda.get_device_capability(0) >= (8, 0) and torch.cuda.is_bf16_supported()
if BACKEND == "auto":
    BACKEND = "bf16" if native_bf16 else "fp16"
if BACKEND not in ("bf16", "fp16") or (BACKEND == "bf16" and not native_bf16):
    raise ValueError("Use fp16, or choose bf16 on a native-BF16 GPU such as L4/A100.")
RUN_DIR = DRIVE / "runs" / f"{RUN_NAME}-{BACKEND}"
print(f"GPU: {torch.cuda.get_device_name(0)} | backend: {BACKEND}")

## 4. Reuse Drive source or clone, then install

In [ ]:
import shlex
import tempfile
from urllib.parse import urlparse

if not ((REPO / "pyproject.toml").is_file() and (REPO / "uv.lock").is_file()):
    if REPO.exists():
        raise FileNotFoundError("Set REPO to a source folder with pyproject.toml and uv.lock.")
    parsed = urlparse(REPO_URL)
    if (
        parsed.scheme != "https"
        or parsed.hostname != "github.com"
        or parsed.port not in (None, 443)
        or parsed.username is not None
        or parsed.password is not None
        or parsed.query
        or parsed.fragment
    ):
        raise ValueError("REPO_URL must be a clean HTTPS GitHub URL.")
    token = optional_secret("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "Private clone needs GITHUB_TOKEN with repository Contents: read access."
        )
    with tempfile.NamedTemporaryFile(
        mode="w", encoding="utf-8", dir="/tmp", delete=False
    ) as askpass_file:
        askpass_file.write("""#!/bin/sh
case "$1" in
  *sername*) printf '%s\n' 'x-access-token' ;;
  *assword*) printf '%s\n' "$GITHUB_TOKEN" ;;
  *) exit 1 ;;
esac
""")
        askpass = Path(askpass_file.name)
    askpass.chmod(0o700)
    os.environ["GITHUB_TOKEN"] = token
    del token
    try:
        !env -u HF_TOKEN \
          GIT_ASKPASS={shlex.quote(str(askpass))} GIT_TERMINAL_PROMPT=0 \
          LC_ALL=C GIT_CONFIG_NOSYSTEM=1 \
          git -c credential.helper= clone --quiet \
          --branch {shlex.quote(REF)} \
          {shlex.quote(REPO_URL)} {shlex.quote(str(REPO))}
        check_shell("Private GitHub clone; check REF, token permissions, and organization approval")
    finally:
        os.environ.pop("GITHUB_TOKEN", None)
        askpass.unlink(missing_ok=True)
assert (REPO / "pyproject.toml").is_file() and (REPO / "uv.lock").is_file()
project = shlex.quote(str(REPO))
!pip install -q uv
check_shell("Install uv")
!uv sync --project {project} --locked
check_shell("uv sync")

## 5. Runtime checks

In [ ]:
!uv run --project {project} --locked chainsaw doctor
check_shell("chainsaw doctor")
!uv run --project {project} --locked chainsaw smoke --backend {shlex.quote(BACKEND)}
check_shell("chainsaw smoke")

## 6. Experiments

Each command writes to this precision-specific Drive folder. `--resume` reuses a nonempty file at the final path; an interrupted run starts over, and a new result is published only after success.

In [ ]:
!uv run --project {project} --locked chainsaw bench-matmul \
  --backend {shlex.quote(BACKEND)} \
  --output {shlex.quote(str(RUN_DIR / 'matmul.csv'))} --resume
check_shell("bench-matmul")

In [ ]:
!uv run --project {project} --locked chainsaw bench-model \
  --backend {shlex.quote(BACKEND)} --model-id {shlex.quote(MODEL_ID)} \
  --output {shlex.quote(str(RUN_DIR / 'model.csv'))} --resume
check_shell("bench-model")

In [ ]:
!uv run --project {project} --locked chainsaw perplexity \
  --backend {shlex.quote(BACKEND)} --model-id {shlex.quote(MODEL_ID)} \
  --max-length 1024 --stride 512 \
  --output {shlex.quote(str(RUN_DIR / 'perplexity.json'))} --resume
check_shell("perplexity")

## 7. Results

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

matmul = pd.read_csv(RUN_DIR / "matmul.csv")
model = pd.read_csv(RUN_DIR / "model.csv")
ppl = json.loads((RUN_DIR / "perplexity.json").read_text())
print(f"WikiText-2 PPL: {ppl['perplexity']:.4f} | {ppl['token_count']} scored tokens")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
matmul.pivot_table(index="m", columns=["n", "k"], values="latency_ms").plot(
    marker="o", ax=axes[0], title=f"{BACKEND.upper()} matmul latency"
)
axes[0].set_ylabel("Latency (ms)")
model.pivot(index="prompt_tokens", columns="phase", values="tokens_per_second").plot(
    kind="bar", ax=axes[1], title="Model throughput"
)
axes[1].set_ylabel("Tokens per second")
fig.tight_layout()
fig.savefig(RUN_DIR / "summary.png", dpi=160)
display(fig)